# Philadelphia 311 Service and Information Requests — Data Acquisition

**Goal of this notebook:** connect to the City of Philadelphia's open data
(served through their CARTO SQL API), confirm the real table/column names,
let you pick a request-type subset based on actual volumes (not guesses),
and save a clean CSV to work with in later notebooks.

Philadelphia's open data (like the shootings dataset, permits, etc.) is served through a
CARTO SQL endpoint at `https://phl.carto.com/api/v2/sql`. We query it directly with SQL
over HTTP — no API key needed for public tables.

**Important:** run this notebook yourself and read the actual output at each step.
City open-data schemas change over time, so Step 1 below verifies the table name and
columns before we trust anything. If a query errors out, the error message from CARTO
usually tells you exactly what's wrong (e.g. an unknown column name) — read it, don't just
rerun blindly.

In [ ]:
import pandas as pd
import requests
import io

CARTO_SQL_URL = "https://phl.carto.com/api/v2/sql"

def carto_query(sql: str) -> pd.DataFrame:
    """Run a SQL query against Philadelphia's public CARTO SQL API and return a DataFrame."""
    resp = requests.get(CARTO_SQL_URL, params={"q": sql, "format": "csv"})
    resp.raise_for_status()
    return pd.read_csv(io.StringIO(resp.text))

## Step 1 — Confirm the table exists and see its real columns

The 311 table is commonly published as `public_cases_fc`, but open-data schemas
get renamed sometimes. This cell checks it directly against `information_schema`
so we're not trusting a name from memory.

In [ ]:
CANDIDATE_TABLE_NAMES = ["public_cases_fc", "311_requests", "service_requests"]

found_table = None
for name in CANDIDATE_TABLE_NAMES:
    try:
        cols = carto_query(
            f"SELECT column_name, data_type FROM information_schema.columns "
            f"WHERE table_name = '{name}' ORDER BY ordinal_position"
        )
        if len(cols) > 0:
            found_table = name
            print(f"Found table: {name}\n")
            print(cols.to_string(index=False))
            break
    except Exception as e:
        print(f"'{name}' failed: {e}")

if found_table is None:
    print(
        "None of the candidate table names worked.\n"
        "Go to https://opendataphilly.org and search '311' to find the current dataset "
        "page — it will list the CARTO table name (or a direct CSV export) under "
        "'API' / 'Data & Resources'. Paste the correct name into CANDIDATE_TABLE_NAMES above."
)

## Step 2 — Peek at a few raw rows

Before writing any real logic, just look at what's actually in the table.

In [ ]:
sample = carto_query(f"SELECT * FROM {found_table} LIMIT 20")
sample.head(20)

## Step 3 — See request-type volumes (this is how we pick the subset)

Adjust `TYPE_COLUMN` below once Step 2 shows you the real column name for the
request/service type (commonly `service_name` or `subject`, but confirm from the
output above).

In [ ]:
TYPE_COLUMN = "service_name"  # <-- confirm/change this against Step 2's output

type_counts = carto_query(
    f"SELECT {TYPE_COLUMN}, COUNT(*) AS n "
    f"FROM {found_table} "
    f"GROUP BY {TYPE_COLUMN} "
    f"ORDER BY n DESC"
)
type_counts

**Decision point:** look at `type_counts` above. Good candidates for our project need:
- decent volume (tens of thousands of rows, ideally, so train/test splits are meaningful)
- a real spread of resolution times (not something that's almost always closed same-day,
  or almost never closed)

Street-related categories the professor suggested (abandoned vehicles, potholes/road
issues, street defects, street lights/signals) are usually mid-to-high volume with
genuine variability in response time, which is exactly what makes a regression /
classification / clustering project non-trivial. Pick the exact string(s) as they
appear in `type_counts` and put them in `CHOSEN_TYPES` below.

In [ ]:
CHOSEN_TYPES = [
    # e.g. "Street Light Outage",
]

assert CHOSEN_TYPES, "Fill in CHOSEN_TYPES with one or more values copied exactly from type_counts above."

## Step 4 — Pull the full subset (paginated)

CARTO's SQL API caps how many rows it returns per request, so we page through with
`LIMIT`/`OFFSET` until we stop getting rows back.

In [ ]:
def fetch_all(table: str, type_col: str, types: list[str], page_size: int = 20000) -> pd.DataFrame:
    types_sql = ", ".join(f"'{t}'" for t in types)
    chunks = []
    offset = 0
    while True:
        sql = (
            f"SELECT * FROM {table} "
            f"WHERE {type_col} IN ({types_sql}) "
            f"ORDER BY objectid "
            f"LIMIT {page_size} OFFSET {offset}"
        )
        chunk = carto_query(sql)
        if len(chunk) == 0:
            break
        chunks.append(chunk)
        print(f"fetched rows {offset} - {offset + len(chunk)}")
        offset += page_size
        if len(chunk) < page_size:
            break
    return pd.concat(chunks, ignore_index=True)

df = fetch_all(found_table, TYPE_COLUMN, CHOSEN_TYPES)
df.shape

If `ORDER BY objectid` errors, Step 2's output will show the real primary-key /
id column name — swap it in.

## Step 5 — Sanity checks + save raw data

Basic shape, dtypes, missingness, and date range — the same first look you'd do on
any new dataset before trusting it.

In [ ]:
print(df.shape)
print(df.dtypes)
print(df.isna().mean().sort_values(ascending=False))

In [ ]:
df.to_csv("philly311_raw.csv", index=False)
print("Saved philly311_raw.csv")

## Next steps (next notebook)
- Parse the created/closed date columns (confirm the real names from Step 2 —
  commonly `requested_datetime` and `closed_datetime`, or `created_date`/`closed_date`)
- Compute `resolution_time = closed - created`
- Handle still-open requests (no closed date) — decide whether to drop them or treat
  them separately; document the choice
- Look at the distribution of resolution_time (it's almost certainly right-skewed —
  decide if/how to transform it)
- Start EDA: resolution time by weekday, by zip/council district, by exact request type